# 07 · EDA Brechas de datos (gaps)

Cuatro sectores del catálogo **no tienen datos físicos** en `data/raw/`:
- `AMBIENTE`, `PARTICIPACION_CIUDADANA` y `SEGURIDAD` — solo contienen `README.md`.
- `SERVICIOS_PUBLICOS` — vacío.

Este notebook documenta qué promete el catálogo, qué indicadores quedan sin construir y
qué se necesita para cerrar la brecha.

## 0. Configuración

In [ ]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")

## 1. Sectores sin datos

In [ ]:
sin_datos = ["AMBIENTE", "PARTICIPACION_CIUDADANA", "SEGURIDAD", "SERVICIOS_PUBLICOS"]
filas = []
for s in sin_datos:
    d = RAW_DIR / s
    archivos = [str(p.relative_to(d)) for p in d.rglob("*") if p.is_file() and p.suffix not in (".md", ".gitkeep")]
    readme = d / "README.md"
    desc = ""
    if readme.exists():
        txt = readme.read_text(encoding="utf-8", errors="ignore").strip()
        desc = txt.splitlines()[0] if txt else ""
    filas.append({"sector": s, "archivos": archivos, "descripcion_readme": desc})
gaps = pd.DataFrame(filas)
display(gaps)
gaps.to_csv(REPORTS / "gaps_sectores_sin_datos.csv", index=False)

## 2. Indicadores que quedan sin construir

In [ ]:
sts = eda.indicator_status()
falt = sts[sts["estado"].astype(str).str.contains("faltante", case=False, na=False)]
display(falt[["indicador", "dimension", "estado", "que_falta"]])
falt.to_csv(REPORTS / "indicadores_faltantes.csv", index=False)
print(sts.groupby("estado").size().to_string())

## 3. Recomendaciones

1. **Conseguir las fuentes prometidas en los README** de AMBIENTE, PARTICIPACIÓN CIUDADANA y
   SEGURIDAD: sin datos físicos no hay indicadores (AMB-01, PAR-01, SEG-01).
2. **SERVICIOS_PUBLICOS** no tiene ni README: definir primero qué se va a medir (cobertura de
   acueducto, alcantarillado, energía, aseo) y de dónde saldrá (SDA, EAAB, Enel, Uaesp).
3. **Indicadores con cruce espacial pendiente** (p. ej. delitos por localidad, calidad de aire
   por localidad): requieren fuentes por localidad o geometrías georreferenciadas.
4. Priorizar el cierre de los indicadores `faltante_territorial` porque el MR ya permite
   georreferenciar cualquier punto a su localidad.

## 4. Tiempos

In [ ]:
guardar_tiempos("07_eda_gaps.csv")